In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GINConv, global_mean_pool
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pickle
import pandas as pd
import random
import os

# === SETUP ===
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === Load Dataset ===
def load_pytorch_dataset(file_path):
    with open(file_path, 'rb') as f:
        dataset = pickle.load(f)
    print(f"Dataset successfully loaded from {file_path}")
    return dataset

pickle_file_path ="C:\Users\nidhi\Downloads\IICT\Pickle files\Random Split\A+B.pkl"
pytorch_dataset = load_pytorch_dataset(pickle_file_path)

# === Model Definition ===
class SimpleGINModelWithGraphGradCAM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate):
        super().__init__()
        self.gin1 = GINConv(nn.Linear(input_dim, hidden_dim))
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.gin2 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.gin3 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.gin4 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn4 = nn.BatchNorm1d(hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.activations = None
        self.gradients = None

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.gin1(x, edge_index)))
        x = F.relu(self.bn2(self.gin2(x, edge_index)))
        x = F.relu(self.bn3(self.gin3(x, edge_index)))
        x = F.relu(self.bn4(self.gin4(x, edge_index)))
        self.activations = x
        x = self.dropout(x)
        x = global_mean_pool(x, batch)
        x = self.fc(x)
        return x

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def register_hooks(self):
        self.gin3.register_full_backward_hook(self.backward_hook)

# === Grad-CAM ===
def graph_grad_cam_with_edges_all(model, data, target_class):
    model.eval()
    data = data.to(device)
    output = model(data)
    target = output[:, target_class]
    target.backward()
    activations = model.activations.detach()
    gradients = model.gradients.detach()
    weights = gradients.mean(dim=1, keepdim=True)
    graph_grad_cam_values = (activations * weights).sum(dim=1)
    graph_grad_cam_values = graph_grad_cam_values / graph_grad_cam_values.max()
    node_positions = torch.arange(graph_grad_cam_values.size(0)).cpu().numpy()
    residue_types = data.x.cpu().numpy()
    node_scores = graph_grad_cam_values.cpu().numpy()
    edge_index = data.edge_index.cpu().numpy()
    edge_scores = [(graph_grad_cam_values[src] + graph_grad_cam_values[dst]) / 2 for src, dst in edge_index.T]
    edge_scores = torch.tensor(edge_scores).cpu().numpy()
    return node_positions, residue_types, node_scores, edge_index.T, edge_scores

# === Train and Evaluate Functions ===
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        data.y = data.y.long()
        output = model(data)
        loss = criterion(output, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        correct += (output.argmax(dim=1) == data.y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, all_preds, all_labels, all_graph_ids = 0, 0, [], [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            data.y = data.y.long()
            output = model(data)
            loss = criterion(output, data.y)
            preds = output.argmax(dim=1)
            total_loss += loss.item() * data.num_graphs
            correct += (preds == data.y).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())
            all_graph_ids.extend([getattr(data, 'graph_id', -1) for data in data.to_data_list()])
    return total_loss / len(loader.dataset), correct / len(loader.dataset), all_preds, all_labels, all_graph_ids

# === Graph ID Assignment ===
for i, data in enumerate(pytorch_dataset, start=1):
    data.graph_id = data.graph_id if hasattr(data, 'graph_id') else i

train_dataset, test_dataset = train_test_split(pytorch_dataset, test_size=0.2, random_state=seed)
train_graph_ids = [data.graph_id for data in train_dataset]
test_graph_ids = [data.graph_id for data in test_dataset]

# === Prepare Output Directory and Initialize Best Epoch ===
quad_folder = r"C:\Users\nidhi\Downloads\GradCAM_A_64"
os.makedirs(quad_folder, exist_ok=True)
best_epoch = 0  # Initialize before using in filenames

# === Save Train and Test Graph Indices ===
train_indices_path = os.path.join(quad_folder, f"train_graph_indices_epoch_{best_epoch}.txt")
test_indices_path = os.path.join(quad_folder, f"test_graph_indices_epoch_{best_epoch}.txt")

with open(train_indices_path, 'w') as f:
    for gid in train_graph_ids:
        f.write(f"{gid}\n")

with open(test_indices_path, 'w') as f:
    for gid in test_graph_ids:
        f.write(f"{gid}\n")

print(f"✅ Train Graph Indices saved: {train_indices_path}")
print(f"✅ Test Graph Indices saved: {test_indices_path}")

# === Initialize Model ===
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model = SimpleGINModelWithGraphGradCAM(20, 64, 2, 0.5).to(device)
model.register_hooks()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

# === Training ===
best_acc = 0
best_preds, best_labels, best_graph_ids = None, None, None

for epoch in range(1, 350):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_loss, test_acc, preds, labels, graph_ids = evaluate(model, test_loader, criterion)

    if 300 <= epoch <= 350 and test_acc > best_acc:
        best_epoch = epoch
        best_acc = test_acc
        best_preds, best_labels, best_graph_ids = preds, labels, graph_ids
        best_model_state = model.state_dict()

    if epoch % 10 == 0 or epoch == 400:
        print(f"Epoch {epoch}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")

# === Save Best Model & Results ===
print(f"\n📌 Best Epoch: {best_epoch} with Test Accuracy: {best_acc:.4f}")
model.load_state_dict(best_model_state)

# === Confusion Matrix ===
cm = confusion_matrix(best_labels, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
plt.figure(figsize=(5, 5))
disp.plot()
plt.title(f"Confusion Matrix (Epoch {best_epoch})")
confusion_path = os.path.join(quad_folder, f"confusion_matrix_epoch_{best_epoch}.png")
plt.savefig(confusion_path)
plt.close()
print(f"✅ Confusion Matrix saved: {confusion_path}")

# === Save Graph Indices by Quadrants ===
tp, fp, tn, fn = [], [], [], []
for i, (pred, label) in enumerate(zip(best_preds, best_labels)):
    gid = best_graph_ids[i]
    if pred == 1 and label == 1:
        tp.append(gid)
    elif pred == 1 and label == 0:
        fp.append(gid)
    elif pred == 0 and label == 0:
        tn.append(gid)
    elif pred == 0 and label == 1:
        fn.append(gid)

def save_indices(name, indices):
    path = os.path.join(quad_folder, f"{name}_indices_epoch_{best_epoch}.txt")
    with open(path, 'w') as f:
        for gid in indices:
            f.write(f"{gid}\n")
    print(f"✅ {name} saved: {path}")

save_indices("True_Positive", tp)
save_indices("False_Positive", fp)
save_indices("True_Negative", tn)
save_indices("False_Negative", fn)

# === Grad-CAM Results ===
for data in test_dataset:
    if data.graph_id in best_graph_ids:
        test_data = data.to(device)
        node_positions, residue_types, node_scores, edge_indices, edge_scores = graph_grad_cam_with_edges_all(
            model, test_data, target_class=1
        )

        node_df = pd.DataFrame({
            "Node_Index": node_positions,
            "Residue_Type": [res.tolist() for res in residue_types],
            "Graph_Grad_CAM_Score": node_scores
        })

        edge_df = pd.DataFrame({
            "Edge_Start": edge_indices[:, 0],
            "Edge_End": edge_indices[:, 1],
            "Graph_Grad_CAM_Edge_Score": edge_scores
        })

        output_file = os.path.join(quad_folder, f"Graph_{data.graph_id}_Grad_CAM_Epoch_{best_epoch}.xlsx")
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            node_df.to_excel(writer, sheet_name="Nodes", index=False)
            edge_df.to_excel(writer, sheet_name="Edges", index=False)
        print(f"✅ Grad-CAM saved: Graph {data.graph_id}")